# 03_Model_Training - Random Forest Regressor

Week 1 Assignment - Stage 2 (Improved Version)

Using Random Forest instead of single Decision Tree - Recommended for better performance

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print("✅ Libraries imported successfully")

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('ds_salaries_cleaned.csv')
print('Total samples:', len(df))
print(df.columns.tolist())

In [ ]:
# Define features and target
target = 'salary_in_usd'

X = df.drop(columns=[target])
y = df[target]

# Feature lists (Best practice for this dataset)
categorical_features = ['employment_type', 'job_category']
numeric_features = ['work_year', 'remote_ratio', 'experience_level_enc', 
                    'company_size_enc', 'is_us_company', 'is_us_residence']

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)

In [ ]:
# 80% Train / 20% Test split (Recommended)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Test set : {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.1f}%)')

In [ ]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Full pipeline with Random Forest
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

print("✅ Pipeline with Random Forest created")

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [10, 15, 20, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

print("Starting Grid Search for Random Forest... This may take a few minutes.")
grid_search.fit(X_train, y_train)

print("\n✅ Best parameters found:")
print(grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)

In [ ]:
# Final evaluation on Test set
best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("\n=== FINAL TEST SET PERFORMANCE - RANDOM FOREST ===")
print(f"Test MAE : ${test_mae:,.0f}")
print(f"Test RMSE: ${test_rmse:,.0f}")
print(f"Test R²  : {test_r2:.4f}")

In [ ]:
# Save the best model
joblib.dump(best_model, 'random_forest_model.joblib')
print("\n✅ Random Forest model saved as 'random_forest_model.joblib'")

In [ ]:
# Feature Importance (Top 15)
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
importances = best_model.named_steps['regressor'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title('Top 15 Most Important Features - Random Forest')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Quick test prediction
sample = X_test.iloc[[0]]
pred = best_model.predict(sample)[0]
actual = y_test.iloc[0]
print(f"\nSample Prediction: Predicted = ${pred:,.0f} | Actual = ${actual:,.0f}")